In [ ]:
import os; os.environ["TRITON_INTERPRET"] = "1" # no cuda available currently
import torch, triton, triton.language as tl

False


relu

In [4]:
@triton.jit()
def kernel_relu(x_ptr, out_ptr, n, BLOCK: tl.constexpr):
	pid = tl.program_id(0)
	offs = pid * BLOCK + tl.arange(0, BLOCK)
	m = offs < n
	x = tl.load(x_ptr + offs, mask=m)
	tl.store(out_ptr + offs, tl.maximum(x, 0), mask=m)
def relu(x):
	out = torch.zeros_like(x)
	n = x.numel()
	grid = (triton.cdiv(n, 1024),)
	kernel_relu[grid](x, out, n, 1024)
	return out

x = torch.randn(10_000)
torch.allclose(torch.relu(x), relu(x))

True

rowwise softmax
of (B, C) batch, embedding etc

In [21]:
@triton.jit()
def kernel_softmax(x_ptr, out_ptr, C, BLOCK: tl.constexpr):
	row = tl.program_id(0)
	cols = tl.arange(0, BLOCK)
	m = cols < C
	offs = row * C + cols
	x = tl.load(x_ptr + offs, mask=m)
	x -= tl.max(x, axis=0)
	x = tl.exp(x)
	x /= tl.sum(x, axis=0)
	tl.store(out_ptr + offs, x, mask=m)
def softmax(x):
	B, C = x.shape
	BLOCK = triton.next_power_of_2(C)
	out = torch.zeros_like(x)
	kernel_softmax[(B,)](x, out, C, BLOCK)
	return out

x = torch.randn(32, 64)
torch.allclose(torch.softmax(x, dim=1), softmax(x))

True